In [4]:
# Class 1: Sports & Athletics (Context: Winning/Medals)
doc1 = "The gold medal price is high effort"
doc2 = "Winning a gold medal needs a high jump"
doc3 = "Market for a gold medal is a trade of sweat"
doc4 = "The athlete will trade all for a gold medal"

# Class 2: Finance & Economy (Context: Market/Investment)
doc5 = "The gold bars price is high today"
doc6 = "Investing in gold bars needs a high rate"
doc7 = "Market for gold bars is a trade of money"
doc8 = "The bank will trade all for gold bars"



In [7]:
pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.9 MB/s eta 0:00:00


In [6]:
pip install  num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 13.9 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=9c956955dbd2fcfbf4c07f75ea74cc626c354acde24f0f50acd85ed27ead313c
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [8]:
import numpy as np
import string
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score
from collections import Counter
import contractions
from num2words import num2words
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

In [9]:


nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Your Task: Fill these functions

def preprocess_text(doc):
    doc = doc.lower()
    doc = contractions.fix(doc)
    num = "".join(c for c in doc if c.isdigit())
    if num:
        doc = doc.replace(num, num2words(int(num)))
    doc = doc.translate(str.maketrans('', '', string.punctuation))
    tokens = doc.split()
    clean_tokens = []
    for t in tokens:
        if t not in stop_words:
            clean_tokens.append(lemmatizer.lemmatize(t))
    return clean_tokens

def vectorize(docs, n_gram_size=1):
    # Implement n-gram extraction and vectorization
    processed_docs = [preprocess_text(doc) for doc in docs]
    all_ngrams = []
    docs_ngrams = []
    for tokens in processed_docs:
        ngrams = []
        for i in range(len(tokens) - n_gram_size + 1):
            gram = ""
            for j in range(n_gram_size):
                gram += tokens[i + j]
                if j != n_gram_size - 1:
                    gram += " "
            ngrams.append(gram)
        docs_ngrams.append(ngrams)
        all_ngrams.extend(ngrams)
    vocab = sorted(list(set(all_ngrams)))
    vectors = []
    for ngrams in docs_ngrams:
        vector = []
        for word in vocab:
            count = 0
            for gram in ngrams:
                if gram == word:
                    count += 1
            vector.append(count)
        vectors.append(vector)
    return np.array(vectors)




[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [10]:
# Training / Clustering

all_docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8]
true_labels = np.array([0,0,0,0,1,1,1,1])
# 1-gram Experiment
X1 = vectorize(all_docs, n_gram_size=1)
km1 = KMeans(n_clusters=2, random_state=42).fit(X1)

# 2-gram Experiment
X2 = vectorize(all_docs, n_gram_size=2)
km2 = KMeans(n_clusters=2, random_state=42).fit(X2)

# Fix cluster label order to dont get 0% accuracy
def align_labels(true, pred):
    if accuracy_score(true, pred) > accuracy_score(true, 1 - pred):
        return pred
    else:
        return 1 - pred

pred1 = align_labels(true_labels, km1.labels_)
pred2 = align_labels(true_labels, km2.labels_)

print(f"1-gram clusters: {km1.labels_}")



# Compare accuracy and precision
acc1 = accuracy_score(true_labels, pred1)
pre1 = precision_score(true_labels, pred1)

acc2 = accuracy_score(true_labels, pred2)
pre2 = precision_score(true_labels, pred2)

print(f"1-gram Accuracy: {acc1}")
print(f"1-gram Precision: {pre1}\n")

print(f"2-gram Accuracy: {acc2}")
print(f"2-gram Precision: {pre2}\n")

if acc2 > acc1 and pre2 > pre1:
    print("2-gram performs better.")
elif acc1 > acc2 and pre1 > pre2:
    print("1-gram performs better.")
else:
    print("Performance is the same for both.")

1-gram clusters: [0 1 0 0 1 1 0 0]
1-gram Accuracy: 0.625
1-gram Precision: 0.6666666666666666

2-gram Accuracy: 0.875
2-gram Precision: 0.8

2-gram performs better.
